# Paso 1: Instalar PySpark

In [1]:
!pip install pyspark

# Paso 2: Subir los archivos a Colab

In [2]:
from google.colab import files
uploaded = files.upload()

Saving movies.tsv to movies.tsv
Saving pdi_sales.csv to pdi_sales.csv
Saving pdi_sales_small.csv to pdi_sales_small.csv


# Paso 3: Inicializar Spark y Actividad 1 (Carga y Parquet)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Inicializamos la sesión de Spark
spark = SparkSession.builder.appName("ActividadesSpark").getOrCreate()

# 1. Crear el esquema de forma declarativa
esquema = StructType([
    StructField("interprete", StringType(), True),
    StructField("pelicula", StringType(), True),
    StructField("anyo", IntegerType(), True)
])

# Cargar el DataFrame indicando el separador de tabulación y el esquema
df_movies = spark.read.option("sep", "\t").schema(esquema).csv("movies.tsv")

# Recuperar el nombre de las columnas
print("Columnas:", df_movies.columns)

# Persistir el dataframe en formato Parquet
df_movies.write.mode("overwrite").parquet("movies.parquet")

# Crear un nuevo dataframe con una muestra de los datos
df_muestra = df_movies.sample(fraction=0.10)
df_muestra.show(5, truncate=False)

Columnas: ['interprete', 'pelicula', 'anyo']
+-------------------+---------------------------------------+----+
|interprete         |pelicula                               |anyo|
+-------------------+---------------------------------------+----+
|Knight, Shirley (I)|Paul Blart: Mall Cop                   |2009|
|Jolie, Angelina    |Alexander                              |2004|
|Jolie, Angelina    |Gone in Sixty Seconds                  |2000|
|Jolie, Angelina    |Mr. & Mrs. Smith                       |2005|
|Cueto, Esteban     |Miss Congeniality 2: Armed and Fabulous|2005|
+-------------------+---------------------------------------+----+
only showing top 5 rows


# Paso 4: Actividad 2 (Transformaciones con datos propios)

In [4]:
from pyspark.sql.functions import col, lit, current_date, year, monotonically_increasing_id

# Datos proporcionados
datos_clientes = [
    { "Nombre": "Aitor", "Edad": 45, "Ciudad": "Elche" },
    { "Nombre": "Marina", "Edad": 14, "Ciudad": "Alicante" },
    { "Nombre": "Laura", "Edad": 19, "Ciudad": "Elche" },
    { "Nombre": "Sonia", "Edad": 45, "Ciudad": "Aspe" },
    { "Nombre": "Pedro", "Edad": None, "Ciudad": "Elche" }
]

# Creamos el DataFrame
df_clientes = spark.createDataFrame(datos_clientes)

# Realizamos las operaciones encadenadas
df_resultado = (df_clientes
    .withColumn("Mayor30", col("Edad") > 30)
    .withColumn("FaltanJubilacion", 67 - col("Edad"))
    .withColumn("Apellidos", lit("XYZ"))
    .drop("Mayor30", "Apellidos")
    .withColumn("AnyoNac", year(current_date()) - col("Edad"))
    .withColumn("Id", monotonically_increasing_id())
    .select("Id", "Nombre", "Edad", "AnyoNac", "FaltanJubilacion", "Ciudad")
)

df_resultado.show()

+----------+------+----+-------+----------------+--------+
|        Id|Nombre|Edad|AnyoNac|FaltanJubilacion|  Ciudad|
+----------+------+----+-------+----------------+--------+
|         0| Aitor|  45|   1981|              22|   Elche|
|         1|Marina|  14|   2012|              53|Alicante|
|8589934592| Laura|  19|   2007|              48|   Elche|
|8589934593| Sonia|  45|   1981|              22|    Aspe|
|8589934594| Pedro|NULL|   NULL|            NULL|   Elche|
+----------+------+----+-------+----------------+--------+



# Paso 5: Actividad 3 (Consultas desde el Parquet)

In [5]:
# Carga del DataFrame desde el formato Parquet generado en el Paso 3
df_parquet = spark.read.parquet("movies.parquet")

print("1. ¿Cuántas películas hay almacenadas? (únicas)")
num_peliculas = df_parquet.select("pelicula").distinct().count()
print(f"Total: {num_peliculas}\n")

print("2. Películas del año 2011:")
df_parquet.filter(col("anyo") == 2011).select("pelicula").distinct().show(truncate=False)

print("3. Películas en las que ha trabajado Murphy, Eddie (I):")
df_parquet.filter(col("interprete") == "Murphy, Eddie (I)").select("pelicula").show(truncate=False)

print("4. Intérpretes que trabajan en la película Superman:")
df_parquet.filter(col("pelicula") == "Superman").select("interprete").show(truncate=False)

print("5. Intérpretes que trabajan en películas que contienen la palabra Scream:")
df_parquet.filter(col("pelicula").contains("Scream")).select("interprete").distinct().show(truncate=False)

print("6. Intérpretes que aparecen tanto en Superman como en Superman II:")
df_sup1 = df_parquet.filter(col("pelicula") == "Superman").select("interprete")
df_sup2 = df_parquet.filter(col("pelicula") == "Superman II").select("interprete")
df_sup1.intersect(df_sup2).show(truncate=False)

1. ¿Cuántas películas hay almacenadas? (únicas)
Total: 1409

2. Películas del año 2011:
+-----------------------------------------+
|pelicula                                 |
+-----------------------------------------+
|Happy Feet Two                           |
|Kung Fu Panda 2                          |
|The Adventures of Tintin                 |
|Dolphin Tale                             |
|Spy Kids: All the Time in the World in 4D|
|Big Mommas: Like Father, Like Son        |
|Source Code                              |
|African Cats                             |
|Winnie the Pooh                          |
|The Adjustment Bureau                    |
|Battle Los Angeles                       |
|Thor                                     |
|The Best Exotic Marigold Hotel           |
|Paranormal Activity 3                    |
|Tower Heist                              |
|New Year's Eve                           |
|Jack and Jill                            |
|Horrible Bosses                